# 04 — Validate, Export & Push Weights to GitHub
Validates, copies and version-tags the trained agent checkpoints from Google Drive
into the repo's `algo_trader/weights/` directory, then commits and pushes them.

**Checkpoint format** (saved by `DeepScalperAgent.save()`):
```
{
  "online_net": state_dict,   # DeepScalperNet weights (BDQ)
  "target_net": state_dict,
  "optimizer":  state_dict,
  "epsilon":    float,
  "steps":      int,
}
```
`strategy.py` supports both this format and a raw `state_dict` (legacy).

**Also committed:** `training_log.csv` produced by `03_train_deepscalper.ipynb`.

**Prerequisites:**
- `GITHUB_TOKEN` must be added to Colab Secrets (🔑 icon) with `repo` scope.
- `GITHUB_USERNAME` and `GITHUB_REPO` must be set in the config cell below.

> ⚠️  This notebook pushes directly to the `main` branch.  
> For production use, push to a `weights` branch and open a PR.

In [ ]:
!pip install -q torch gitpython

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

# ─── EDIT THESE ──────────────────────────────────────────────────────────────
GITHUB_USERNAME = 'YOUR_GITHUB_USERNAME'
GITHUB_REPO     = 'deepscalper_copilot'
# ─────────────────────────────────────────────────────────────────────────────

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
if not GITHUB_TOKEN:
    raise RuntimeError('Add GITHUB_TOKEN (with repo scope) to Colab Secrets.')

import os, sys

DRIVE_WEIGHTS  = '/content/drive/MyDrive/algo_trader/weights'
DRIVE_LOG      = '/content/drive/MyDrive/algo_trader/training_log.csv'
REPO_DIR       = '/content/deepscalper_copilot'
REPO_WEIGHTS   = f'{REPO_DIR}/algo_trader/weights'

os.makedirs(REPO_WEIGHTS, exist_ok=True)

# Make the repo importable so we can validate checkpoints with DeepScalperNet
ALGO_DIR = REPO_DIR + '/algo_trader'
if ALGO_DIR not in sys.path:
    sys.path.insert(0, ALGO_DIR)

print('Paths configured ✓')

In [ ]:
import git

REMOTE_URL = (
    f'https://{GITHUB_USERNAME}:{GITHUB_TOKEN}'
    f'@github.com/{GITHUB_USERNAME}/{GITHUB_REPO}.git'
)

if os.path.exists(REPO_DIR + '/.git'):
    repo = git.Repo(REPO_DIR)
    print('Repo already cloned — pulling latest changes...')
    origin = repo.remote('origin')
    origin.set_url(REMOTE_URL)
    origin.pull('main')
else:
    print('Cloning repo...')
    repo = git.Repo.clone_from(REMOTE_URL, REPO_DIR)

print(f'HEAD: {repo.head.commit.hexsha[:8]}  ({repo.head.commit.message.strip()})')

In [ ]:
import shutil
import torch
from pathlib import Path
from colab.deepscalper.architecture import DeepScalperNet

pth_files = sorted(Path(DRIVE_WEIGHTS).glob('*.pth'))
print(f'Found {len(pth_files)} weight file(s) in Drive.\n')

copied  = []
skipped = []

for src in pth_files:
    # ── Validate checkpoint format ────────────────────────────────────────────
    try:
        ckpt = torch.load(str(src), map_location='cpu', weights_only=True)
    except Exception as e:
        print(f'  SKIP {src.name}: could not load — {e}')
        skipped.append(src.name)
        continue

    # Support both agent checkpoint (has "online_net") and raw state_dict
    state_dict = ckpt.get('online_net', ckpt)

    try:
        net = DeepScalperNet()
        net.load_state_dict(state_dict, strict=True)
    except Exception as e:
        print(f'  SKIP {src.name}: incompatible with DeepScalperNet — {e}')
        skipped.append(src.name)
        continue

    dst = Path(REPO_WEIGHTS) / src.name
    shutil.copy2(src, dst)
    copied.append(str(dst))
    print(f'  OK   {src.name}')

print(f'\nCopied {len(copied)} / {len(pth_files)} files → {REPO_WEIGHTS}')
if skipped:
    print(f'Skipped: {skipped}')

In [ ]:
from datetime import datetime

if not copied:
    raise RuntimeError('No valid weight files to commit. Check the validation output above.')

# Stage .pth weight files
staged_paths = [str(Path(f).relative_to(REPO_DIR)) for f in copied]
repo.index.add(staged_paths)

# Stage training_log.csv if present on Drive
log_dst = Path(REPO_DIR) / 'algo_trader' / 'training_log.csv'
if os.path.exists(DRIVE_LOG):
    shutil.copy2(DRIVE_LOG, log_dst)
    repo.index.add([str(log_dst.relative_to(REPO_DIR))])
    print(f'Staged training_log.csv ({log_dst})')

if repo.is_dirty():
    timestamp  = datetime.utcnow().strftime('%Y-%m-%d-%H%M')
    tag_name   = f'weights-{timestamp}'
    commit_msg = (
        f'chore: push DeepScalper weights [{timestamp}] '
        f'({len(copied)} ticker(s))'
    )

    with repo.config_writer() as cfg:
        cfg.set_value('user', 'name',  GITHUB_USERNAME)
        cfg.set_value('user', 'email', f'{GITHUB_USERNAME}@users.noreply.github.com')

    commit = repo.index.commit(commit_msg)
    print(f'Committed: {commit.hexsha[:8]} — {commit_msg}')

    repo.create_tag(tag_name, ref=commit)
    print(f'Tag created: {tag_name}')

    origin = repo.remote('origin')
    origin.push('main')
    origin.push(tag_name)
    print(f'\n✅ Pushed to github.com/{GITHUB_USERNAME}/{GITHUB_REPO}  (tag: {tag_name})')
else:
    print('Nothing new to commit — repo is already up to date.')